<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 5. Мультиагентная система — подробная методичка

Эта методичка продолжает Этапы 1–4. Предполагается, что у вас:
- `docker compose ps` — `db` и `app` в `healthy`;
- `/health` возвращает 200 с `database: healthy`;
- `/auth/register`, `/auth/login`, `/auth/me` работают;
- 54 теста Этапов 2–3 + тесты Этапа 4 зелёные;
- `backend/db/models.py` содержит `User`, `GlossaryDB`, `TMDB`, `UncertainExample`, `TranslationCache`.

Если чего-то нет — вернитесь к Этапу 4.

**Цель этапа:** Supervisor + Validator + Critic + Editor. LLM пока не вызывается реально — у `Critic._call_llm` и `Editor._call_llm` временные заглушки. Реальные вызовы появятся в Этапе 8.

**Недель:** 2. Это самый большой этап по количеству кода.

**Что вы получите в конце этапа:**

- `backend/utils/async_utils.py` — `@retry(max_attempts, delay)` декоратор.
- `backend/utils/http_client.py` — per-loop `httpx.AsyncClient` + `close_ollama_clients()`.
- `backend/utils/hf_llm.py` — `get_hf_llm`/`unload_hf_llm` фабрика (полная реализация, но не вызывается до Этапа 8).
- `backend/utils/llm_processing.py` — `clean_translation` (очистка ответов LLM).
- `backend/agents/base.py` — `BaseAgent(ABC)` и `Agent(BaseAgent)` с `_resolve_engine`, `_call_llm` под `@retry`, `seed=42`.
- `backend/agents/supervisor.py` — оркестратор с `_adaptive_budget`, `_snapshot`/`_restore`, `_is_improvement`, `_compute_final`.
- `backend/agents/validator.py` — rule-based, 7 проверок, LLM не вызывает.
- `backend/agents/critic.py` — LLM-as-a-Judge, трёхуровневый few-shot, сканер JSON, fallback `failed=True`.
- `backend/agents/editor.py` — LLM-редактор, пропуск при `critic_failed` и при отсутствии issues.

**Критерии готовности:**

- [ ] `BaseAgent` абстрактный, `process` обязателен; `Agent` резолвит `engine` из `auto | ollama | huggingface`.
- [ ] `_call_llm` пиннит `seed=42`.
- [ ] `_adaptive_budget` возвращает 3/2/1/0 по правилам.
- [ ] `_snapshot`/`_restore` реализованы, `reasoning_log` не откатывается.
- [ ] `_is_improvement` возвращает `False` при регрессии score, при `critic_failed`, при откате `validation_passed`.
- [ ] Critic парсит JSON сканером с балансировкой скобок; при ошибке — `overall_score=None`, `failed=True` (**не 5.0**).
- [ ] Editor не вызывает LLM при `critic_failed` и при отсутствии issues.
- [ ] Все четыре агента покрыты unit-тестами (≥ 15 сценариев).
- [ ] Ветка `week-5` создана, коммит сделан.

> **Важно.** Этот этап создаёт 4 вспомогательных утилиты, которые «принадлежат» Этапу 8 по смыслу (LLM), но нужны для импорта уже сейчас. `_call_llm` в `Critic` и `Editor` **временно переопределяется на заглушку** — это единственное отклонение от финального кода, которое вы снимете в Этапе 8. Всё остальное — 1:1 финальный код.

---

## 5.0. Подготовка: ветка и проверка зависимостей

```powershell
cd D:\RUNG
git checkout -b week-5
```

Проверьте, что в `pyproject.toml` есть:

```toml
httpx = "^0.28.1"
torch = "^2.13.0"
transformers = "^5.15.1"
```

`torch`/`transformers` нужны для `hf_llm.py` (полная реализация создаётся здесь, но не вызывается до Этапа 8). Импорт модуля не грузит модель — только регистрирует классы.

---

## 5.1. `backend/utils/async_utils.py` — декоратор `retry`

**Что это:** `@retry(max_attempts, delay)` для async-функций. Оборачивает функцию так, что при исключении она повторяется N раз с задержкой.

**Зачем это нужно:** `_call_llm` бросает `httpx.TimeoutException` или `ConnectError`. Ollama на холодную загрузку 14B-модели может ответить за 30+ секунд. Один retry спасает от «первый запрос всегда падает», второй — от транзиентного сбоя сети, третий — от гонки при рестарте Ollama.

Создайте `backend/utils/async_utils.py`:

```python
# backend/utils/async_utils.py
import asyncio
import logging
from functools import wraps
from typing import Any, Callable

logger = logging.getLogger(__name__)


def retry(max_attempts: int = 3, delay: float = 1.0):
    """Decorator to retry an async function on exceptions."""
    if max_attempts < 1:
        raise ValueError("retry: max_attempts must be >= 1")

    def decorator(func: Callable) -> Callable:
        @wraps(func)
        async def wrapper(*args, **kwargs) -> Any:
            for attempt in range(max_attempts):
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    if attempt < max_attempts - 1:
                        logger.warning(
                            "Attempt %d/%d for %s failed: %s. "
                            "Retrying in %ss...",
                            attempt + 1,
                            max_attempts,
                            func.__name__,
                            e,
                            delay,
                        )
                        await asyncio.sleep(delay)
                    else:
                        logger.error(
                            "All %d attempts failed for %s",
                            max_attempts,
                            func.__name__,
                        )
                        raise

        return wrapper

    return decorator
```

### Разбор

**Почему `max_attempts < 1` бросает `ValueError` на этапе создания декоратора.** Это ошибка программиста, а не runtime. Если `retry(max_attempts=0)` попадёт в код, без проверки цикл `for attempt in range(0)` не выполнится, и функция молча вернёт `None` — худший баг, чем падение. Проверка в `decorator`-фабрике — fail fast.

**Почему `@wraps(func)`.** Сохраняет `__name__`, `__doc__`, `__module__` — без этого `func.__name__` в логе было бы `"wrapper"`, а не `"_call_llm"`. Плюс некоторые фреймворки (FastAPI, pytest) различают функции по имени.

**Почему `except Exception`, а не конкретные типы.** Мы хотим ретраить любой сбой LLM: timeout, HTTP 500, connection reset, OOM. Если ограничиться `httpx.TimeoutException`, остальные ошибки пройдут мимо и не повторятся. Ловим всё, что не `BaseException` (не `KeyboardInterrupt`, не `SystemExit`).

**Почему `await asyncio.sleep(delay)` перед последней попыткой.** После первой неудачи — 1 секунда, после второй — ещё 1 секунда, после третьей — `raise`. Итого 2 секунды паузы на 3 попытки. Если Ollama перегружена — второй запрос через секунду обычно проходит.

**Почему не экспоненциальный backoff.** Для 3 попыток с задержкой 1 с экспонента не даёт выигрыша. `OLLAMA_KEEP_ALIVE=30m` в compose держит модель в памяти, и 1 секунды достаточно, чтобы освободился слот `OLLAMA_NUM_PARALLEL=2`. Экспонента нужна для rate limiting внешних API — здесь не тот случай.

### Коммит

```powershell
git add backend/utils/async_utils.py
git commit -m "feat(week-5): add retry decorator for async functions"
```

---

## 5.2. `backend/utils/http_client.py` — per-loop httpx-клиент

**Что это:** единый `httpx.AsyncClient` на каждый event loop, кэшированный в `WeakKeyDictionary`. Плюс `close_ollama_clients()` для shutdown.

**Зачем это нужно:** `httpx.AsyncClient` держит пул соединений. Если создавать его на каждый запрос — TCP handshake + TLS (если HTTPS) на каждый вызов, что убивает производительность на горячем пути. Если шарить один клиент между разными event loop'ами (FastAPI и Celery) — соединения привязаны к loop'у, что даёт `RuntimeError: Event loop is closed`. Решение — один клиент **на loop**, зарегистрированный в слабой ссылке.

Создайте `backend/utils/http_client.py`:

```python
# backend/utils/http_client.py
import asyncio
import logging
import threading
import weakref

import httpx

logger = logging.getLogger(__name__)

_DEFAULT_TIMEOUT = 300.0

_clients: "weakref.WeakKeyDictionary[asyncio.AbstractEventLoop, httpx.AsyncClient]" = (
    weakref.WeakKeyDictionary()
)
_lock = threading.Lock()


def get_ollama_client() -> httpx.AsyncClient:
    """
    Return an ``httpx.AsyncClient`` bound to the current event loop.

    One client per loop keeps the connection pool reusable within a
    single loop while avoiding sharing sockets across loops — FastAPI
    and the Celery worker each run their own.
    """
    loop = asyncio.get_running_loop()
    with _lock:
        client = _clients.get(loop)
        if client is None:
            client = httpx.AsyncClient(timeout=_DEFAULT_TIMEOUT)
            _clients[loop] = client
            logger.debug("Created httpx.AsyncClient for loop id=%s", id(loop))
        return client


async def close_ollama_clients() -> None:
    with _lock:
        clients = list(_clients.values())
        _clients.clear()
    for client in clients:
        try:
            await client.aclose()
        except Exception as e:
            logger.warning("Error closing httpx.AsyncClient: %s", e)
```

### Разбор

**Почему `_DEFAULT_TIMEOUT = 300.0`, а не 30 или 60.** Это совпадает с `stop_grace_period: 300s` в `docker-compose.yml` и `proxy_read_timeout 600s` в `nginx.conf`. Худший сценарий — холодный старт 14B-модели + длинный промпт Critic + Editor, это может занять 2–3 минуты. 30 с обрежут легитимные запросы; 300 с — с запасом.

**Почему `weakref.WeakKeyDictionary`, а не `dict`.** Loop'ы создаются и умирают (в тестах — каждый тест новый loop в `pytest-asyncio`). Обычный `dict` держал бы мёртвые loop'ы в памяти вечно — утечка. Слабая ссылка позволяет loop'у умереть, а клиенту — освободиться при GC.

**Почему `threading.Lock`, а не `asyncio.Lock`.** `get_ollama_client()` — синхронная функция (её нельзя `await` внутри FastAPI, она вызывается из `_call_llm` до `await`). Внутри одного loop'а конкурентность однопоточная (GIL), но `get_running_loop` + `_clients.get` могут выполняться в разных потоках (FastAPI threadpool для sync-эндпоинтов). `threading.Lock` защищает от гонки при параллельном создании.

**Почему `asyncio.get_running_loop()`, а не `asyncio.get_event_loop()`.** В Python 3.10+ `get_event_loop()` без запущенного loop бросает `DeprecationWarning` и в 3.12 — `RuntimeError`. `get_running_loop()` явно требует активный loop — правильный контракт для async-функции (её всегда вызывают из корутины).

**Почему `close_ollama_clients` — async.** `client.aclose()` — корутина. Вызывается из `lifespan` в `main.py` (Этап 8) и из `_shutdown_worker_resources` в `celery_app.py` (Этап 11).

**Почему `try/except` вокруг `aclose()`.** Если клиент уже закрыт — `aclose()` идемпотентен, но может бросить при повторном вызове на некоторых версиях httpx. Плюс если `close_ollama_clients` вызывается дважды (shutdown fastapi + shutdown celery), второй вызов должен быть no-op.

### Коммит

```powershell
git add backend/utils/http_client.py
git commit -m "feat(week-5): add per-loop httpx client and shutdown"
```

---

## 5.3. `backend/utils/hf_llm.py` — фабрика HF-моделей

**Что это:** обёртка над `transformers` для генерации текста. `get_hf_llm(model_name)` возвращает singleton на модель, `unload_hf_llm(model_name=None)` сбрасывает.

**Зачем это нужно уже на Этапе 5:** `critic.py` и `editor.py` в `__init__` импортируют `get_hf_llm`. Если файла нет — `ImportError` при импорте `Critic`, и всё приложение не стартует. Создаём файл целиком сейчас; вызов `get_hf_llm` произойдёт только при `engine="huggingface"` — в `.env` стоит `AGENT_ENGINE=ollama`, значит, HF не загружается.

Создайте `backend/utils/hf_llm.py` — полный код из референсного проекта:

```python
# backend/utils/hf_llm.py
import asyncio
import logging
from typing import Dict, Optional

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

from backend.core.config import get_settings

logger = logging.getLogger(__name__)

_DEFAULT_SEED = 42


class HuggingFaceLLM:
    """Async wrapper around a Hugging Face model for text generation."""

    def __init__(self, model_name: str):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.model_type: Optional[str] = None
        self.device = self._pick_device()
        self._load_lock = asyncio.Lock()
        self._loaded = False

    @staticmethod
    def _pick_device() -> str:
        if torch.cuda.is_available():
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
        return "cpu"

    def _pick_dtype(self) -> torch.dtype:
        if self.device == "cuda":
            return torch.float16
        return torch.float32

    async def _load_model(self) -> None:
        if self._loaded:
            return

        async with self._load_lock:
            if self._loaded:
                return

            settings = get_settings()
            logger.info(
                "Loading Hugging Face model '%s' on device=%s ...",
                self.model_name, self.device,
            )
            loop = asyncio.get_running_loop()

            def _load():
                tokenizer = AutoTokenizer.from_pretrained(
                    self.model_name,
                    token=settings.HF_TOKEN,
                    trust_remote_code=False,
                )

                try:
                    model = AutoModelForCausalLM.from_pretrained(
                        self.model_name,
                        token=settings.HF_TOKEN,
                        trust_remote_code=False,
                        torch_dtype=self._pick_dtype(),
                    )
                    return tokenizer, model, "causal"
                except (ValueError, OSError) as e:
                    logger.info(
                        "'%s' is not loadable as CausalLM "
                        "(%s: %s); trying Seq2Seq",
                        self.model_name, type(e).__name__, e,
                    )
                    model = AutoModelForSeq2SeqLM.from_pretrained(
                        self.model_name,
                        token=settings.HF_TOKEN,
                        trust_remote_code=False,
                        torch_dtype=self._pick_dtype(),
                    )
                    return tokenizer, model, "seq2seq"

            try:
                tokenizer, model, model_type = await loop.run_in_executor(
                    None, _load
                )
            except Exception:
                logger.exception(
                    "Failed to load model '%s'", self.model_name
                )
                raise

            model.eval()
            try:
                model.to(self.device)
            except Exception as e:
                logger.warning(
                    "Could not move '%s' to %s (%s); staying on CPU",
                    self.model_name, self.device, e,
                )
                self.device = "cpu"

            self.tokenizer = tokenizer
            self.model = model
            self.model_type = model_type
            self._loaded = True
            logger.info(
                "Model '%s' loaded (type=%s, device=%s)",
                self.model_name, model_type, self.device,
            )

    @staticmethod
    def _seed_everything(seed: int) -> None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    async def generate(
        self,
        prompt: str,
        max_new_tokens: int = 512,
        temperature: float = 0.0,
        seed: int = _DEFAULT_SEED,
    ) -> str:
        await self._load_model()
        loop = asyncio.get_running_loop()

        def _generate() -> str:
            if seed is not None:
                self._seed_everything(seed)

            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            generate_kwargs = {
                "max_new_tokens": max_new_tokens,
                "do_sample": temperature > 0.0,
                "pad_token_id": self.tokenizer.pad_token_id,
            }
            if temperature > 0.0:
                generate_kwargs["temperature"] = temperature

            with torch.inference_mode():
                outputs = self.model.generate(**inputs, **generate_kwargs)

            result = self.tokenizer.decode(
                outputs[0], skip_special_tokens=True
            )

            if self.model_type == "causal" and prompt in result:
                result = result.split(prompt)[-1].strip()

            return result

        return await loop.run_in_executor(None, _generate)


_instances: Dict[str, HuggingFaceLLM] = {}


def get_hf_llm(model_name: str) -> HuggingFaceLLM:
    instance = _instances.get(model_name)
    if instance is None:
        instance = HuggingFaceLLM(model_name)
        _instances[model_name] = instance
        logger.debug("Created HuggingFaceLLM singleton for '%s'", model_name)
    return instance


def unload_hf_llm(model_name: Optional[str] = None) -> None:
    global _instances
    if model_name is None:
        _instances.clear()
        logger.info("All HuggingFaceLLM instances cleared")
    else:
        _instances.pop(model_name, None)
        logger.info("HuggingFaceLLM instance for '%s' cleared", model_name)
```

### Разбор

**Почему `_load_model` под `asyncio.Lock`.** Два параллельных запроса к `/translate` при `engine="huggingface"` могут одновременно дойти до `_load_model`. Без локa оба начнут загружать модель — 2× RAM и race condition в `_instances`. С `Lock` — второй ждёт, пока первый загрузит, и получает готовую.

**Почему `model.to(self.device)` в `try/except`.** MPS (Apple Silicon) иногда не поддерживает оператор, который модель использует, и падает при переносе. CUDA может не иметь памяти. Без `try/except` — приложение падает при первом запросе; с ним — модель остаётся на CPU и работает (медленнее).

**Почему `torch.float16` только на CUDA.** fp16 на CPU не поддерживается частью операторов `torch`; на MPS — нестабильно для некоторых моделей. fp32 на CPU/MPS — медленнее, но работает всегда.

**Почему `run_in_executor(None, _generate)`.** `model.generate()` — синхронная блокирующая операция на десятки секунд. Без `run_in_executor` она блокирует event loop, FastAPI не может обслуживать другие запросы. Threadpool executor — стандартный паттерн для CPU-bound/generation в async.

**Почему `if self.model_type == "causal" and prompt in result`.** Causal LM (`Qwen`, `Llama`) возвращает `prompt + generated`. Seq2Seq (`NLLB`, `T5`) — только `generated`. Для causal нужно отрезать prompt, иначе в ответе будет `"Translate from en to ru: ...\n\nTranslation: переведённый текст"` вместо просто перевода.

**Почему `_instances: Dict[str, HuggingFaceLLM]`, а не `WeakKeyDictionary`.** Модель — тяжёлый объект (несколько ГБ RAM). Её нельзя выгружать автоматически по GC — это приведёт к перезагрузке на каждый запрос. Singleton держит модель до `unload_hf_llm()` (явный вызов) или до конца процесса.

**Почему `unload_hf_llm` нужна уже сейчас.** В тестах (раздел 5.8) `Critic(engine="huggingface")` создаст `HuggingFaceLLM`. Без `unload_hf_llm()` между тестами singleton остаётся в `_instances`, ссылки на модель живут в памяти, `pytest` съедает RAM. `unload_hf_llm()` в фикстуре `autouse` — обязательный приём.

### Коммит

```powershell
git add backend/utils/hf_llm.py
git commit -m "feat(week-5): add HuggingFace LLM wrapper with singleton registry"
```

---

## 5.4. `backend/utils/llm_processing.py` — `clean_translation`

**Что это:** очистка «сырого» ответа LLM от обёрток, префиксов, JSON-артефактов, CJK-пунктуации.

**Зачем это нужно:** LLM (Qwen, NLLB, Ollama) регулярно отвечает не чистым переводом, а:
- `{"translation": "привет"}` (JSON)
- ` ```json\n{"translation": "привет"}\n``` ` (Markdown)
- `Translation: привет` (префикс)
- `Here is the translation: привет` (meta)
- `привет，мир。` (CJK-пунктуация вместо `.` и `,`)

`Editor` вызывает `clean_translation(raw_corrected)` перед сохранением. В Этапе 9 `perform_translation` тоже очищает финальный перевод перед кэшированием.

Создайте `backend/utils/llm_processing.py`:

```python
# backend/utils/llm_processing.py
import json
import logging
import re

logger = logging.getLogger(__name__)

_TOKENIZER_ARTEFACT_RE = re.compile(
    r"(?:LBLBLB|YYZBB|CCWCC|XXZXX)[A-Z_]*",
    re.IGNORECASE,
)

_CJK_RE = re.compile(r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff]")

_CJK_PUNCT_RE = re.compile(r"[\u3000-\u303F\uFF00-\uFFEF]")


def clean_translation(raw: str) -> str:
    if not raw:
        return ""

    original = raw

    # 1. Single JSON object
    try:
        data = json.loads(raw)
        if isinstance(data, dict):
            extracted = data.get("translation", data.get("text", raw))
            if isinstance(extracted, str):
                raw = extracted.strip()
            else:
                logger.warning("JSON value is not a string, keeping original response")
        else:
            logger.debug("JSON is not an object, keeping original response")
    except (json.JSONDecodeError, AttributeError):
        logger.debug("Response is not a single JSON object, trying other formats")

    # 2. ```json ... ``` fenced block
    json_match = re.search(r"```json\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group(1))
            extracted = data.get("translation", data.get("text", raw))
            if isinstance(extracted, str):
                raw = extracted.strip()
        except (json.JSONDecodeError, AttributeError) as e:
            logger.debug("Failed to parse JSON block: %s", e)

    # 3. XML wrapper
    xml_match = re.search(r"<[^>]+>(.*?)</[^>]+>", raw, re.DOTALL)
    if xml_match:
        raw = xml_match.group(1).strip()

    # 4. Leading "Translation:" / "Перевод:" style prefixes
    prefixes = [
        r"^Translation\s*\([^)]*\)\s*:\s*",
        r"^Translation\s*:\s*",
        r"^Перевод\s*:\s*",
        r"^Ответ\s*:\s*",
        r"^Here is the translation\s*:\s*",
        r"^Result\s*:\s*",
    ]
    for p in prefixes:
        raw = re.sub(p, "", raw, flags=re.IGNORECASE)

    # 5. Meta lines ("Here is...", "Sure,...", "As an AI,...")
    lines = raw.split('\n')
    filtered_lines = []
    for line in lines:
        if re.match(
            r'^(I am|Note:|Here is|Let me|Sure|Of course|As an AI|I\'ll|I will|My response|The translation is)',
            line.strip(),
            re.IGNORECASE
        ):
            continue
        filtered_lines.append(line)
    raw = '\n'.join(filtered_lines).strip()

    # 6. Surrounding quotes
    raw = raw.strip(" \"'")

    # 7. Tokenizer artefacts (LBLBLB_YYZBB_... chains)
    if _TOKENIZER_ARTEFACT_RE.search(raw):
        logger.warning("Tokenizer artefact detected in output")
        raw = _TOKENIZER_ARTEFACT_RE.sub("", raw).strip()

    # 8. Leaked CJK characters in non-CJK text
    if _CJK_RE.search(raw):
        logger.warning("CJK characters detected in output")
        raw = _CJK_RE.sub("", raw).strip()

    # 9. Leaked CJK punctuation in non-CJK text
    if _CJK_PUNCT_RE.search(raw):
        logger.warning("CJK punctuation detected in output")
        raw = _CJK_PUNCT_RE.sub("", raw).strip()

    if not raw:
        logger.warning("Clean translation resulted in empty string, returning raw")
        return original.strip()

    return raw
```

### Разбор

**Почему `original = raw` в начале и `return original.strip()` в конце.** Если очистка убила весь текст (например, ответ был `"，。"` — только CJK-пунктуация), лучше вернуть исходный ответ, чем пустую строку. Пустая строка в `Editor` = `editor_failed=True` = откат. Грязный ответ лучше, чем никакого — `Critic` (Этап 8) потом поставит низкий балл, `Editor` попробует исправить.

**Почему сначала `json.loads(raw)`, потом regex на fenced.** Прямой JSON — быстрый путь. Fenced-блок — частая обёртка (` ```json ... ``` `). Regex `re.DOTALL` — потому что JSON может быть многострочным.

**Почему `data.get("translation", data.get("text", raw))`.** Разные LLM используют разные ключи. Ollama/qwen — `"response"`, но если промпт просит `{"translation": ...}`, ответ будет с этим ключом. Fallback на `"text"` — для шаблонов, которые просят `{"text": ...}`. Если ничего нет — вернуть `raw` (JSON не подошёл).

**Почему режем `Translation:` и `Перевод:` без `^`.** Regex с `^` привязан к началу строки. Если LLM ответил `"Here is the translation: привет"` — `^Translation:` не сработает, а `^Here is the translation:` сработает. Порядок в `prefixes` важен: сначала более специфичные (`Translation (...):` с параметрами в скобках), потом общие (`Translation:`).

**Почему `line.startswith(...)` через `re.match`.** `re.match` привязан к началу. `re.search` нашёл бы паттерн в середине строки и удалил бы кусок текста. `re.match` — безопаснее.

**Почему `_TOKENIZER_ARTEFACT_RE = r"(?:LBLBLB|YYZBB|CCWCC|XXZXX)[A-Z_]*"`.** Это конкретные последовательности, которые NLLB и qwen выдают на редких Unicode (Quranic marks, диакритика). Например, `"LBLBLB_YYZBB_1"`. Паттерн ловит токен и всё, что идёт за ним из заглавных и подчёркиваний — до первого пробела или строчной.

**Почему `_CJK_RE` и `_CJK_PUNCT_RE` — разные регексы.** `_CJK_RE` — иероглифы (китайские, японские), которые в переводе на русский — явная ошибка. `_CJK_PUNCT_RE` — пунктуация `，` `。` `：` `；` `！` `？`, которая **часто** вылезает у qwen даже в правильном по смыслу русском тексте. Их чистят отдельно, чтобы лог различал «LLM выдал иероглифы» и «LLM использовал китайскую запятую».

**Почему комментарий про «if CJK target language ever added».** Сейчас модуль language-agnostic, чистит CJK всегда. Если в `LANGUAGES` добавят `zh`/`ja`, `clean_translation` сломает перевод на китайский. Пометка в коде — предупреждение.

### Проверка

```powershell
python -m poetry run python -c "from backend.utils.llm_processing import clean_translation; print(clean_translation('{\"translation\": \"привет，мир。\"}'))"
```

**Что вы должны увидеть:**

```
приветмир
```

CJK-пунктуация удалена, JSON развёрнут. (Пробел пропал — `，` → `.sub("")` без замены на пробел; это ограничение, но так в референсном коде.)

### Коммит

```powershell
git add backend/utils/llm_processing.py
git commit -m "feat(week-5): add clean_translation (JSON, prefixes, CJK, artefacts)"
```

---

## 5.5. `backend/agents/base.py` — `BaseAgent` и `Agent`

**Что это:** два класса.

- `BaseAgent(ABC)` — интерфейс агента: `process(state)` абстрактный, `name`, `_last_result`, `get_last_result()`. Наследуется `Validator` (rule-based).
- `Agent(BaseAgent)` — LLM-backed агент. Резолвит `engine`, реализует `_call_llm` с `retry`, `seed=42`. Наследуются `Critic` и `Editor`.

Создайте `backend/agents/base.py`:

```python
# backend/agents/base.py
import logging
from abc import ABC, abstractmethod
from typing import Any, Dict, Optional

from backend.core.config import get_settings
from backend.core.exceptions import ModelNotLoadedError
from backend.utils.async_utils import retry
from backend.utils.http_client import get_ollama_client

logger = logging.getLogger(__name__)

_DEFAULT_SEED = 42


class BaseAgent(ABC):
    """
    Abstract base for every agent in the pipeline.

    Exposes the two methods the Supervisor drives every stage through:
    ``process(state)`` and ``get_last_result()``.
    """

    def __init__(self, name: Optional[str] = None):
        self.name = name or self.__class__.__name__
        self._last_result: Any = None

    @abstractmethod
    async def process(self, state: Dict[str, Any]) -> Dict[str, Any]:
        """Process the current state and return an updated state."""

    def get_last_result(self) -> Any:
        return self._last_result


class Agent(BaseAgent):
    """
    Base for LLM-backed agents.

    Subclasses declare which Settings fields hold their HF- and
    Ollama-model names via ``hf_model_setting`` / ``ollama_model_setting``.
    Setting either to ``None`` means the agent does not support the
    corresponding engine.

    The engine is resolved once in ``__init__`` and always normalizes to
    a concrete value: ``"ollama"`` or ``"huggingface"``.

    Every call to ``_call_llm`` pins ``seed=42``. The Editor runs at
    temperature 0.3, so without a fixed seed the same input produces a
    different correction on every run and eval modes that include the
    Editor are not reproducible.
    """

    hf_model_setting: Optional[str] = None
    ollama_model_setting: Optional[str] = None

    model_name: Optional[str] = None
    hf_llm: Any = None
    temperature: float = 0.0
    max_new_tokens: int = 512

    def __init__(self, name: Optional[str] = None, engine: Optional[str] = None):
        super().__init__(name=name)
        self.settings = get_settings()
        requested = engine or self.settings.AGENT_ENGINE
        self.engine = self._resolve_engine(requested)

    def _hf_model_name(self) -> Optional[str]:
        if not self.hf_model_setting:
            return None
        value = getattr(self.settings, self.hf_model_setting, None)
        if not value or not str(value).strip():
            return None
        return str(value).replace("ollama/", "")

    def _ollama_model_name(self) -> Optional[str]:
        if not self.ollama_model_setting:
            return None
        value = getattr(self.settings, self.ollama_model_setting, None)
        if not value or not str(value).strip():
            return None
        return str(value).replace("ollama/", "")

    def _resolve_engine(self, requested: str) -> str:
        requested = (requested or "").lower()

        if requested == "huggingface":
            if self._hf_model_name():
                return "huggingface"
            logger.warning(
                "[%s] engine=huggingface but %s is empty — falling back to ollama",
                self.name,
                self.hf_model_setting,
            )
            return "ollama"

        if requested == "auto":
            return "huggingface" if self._hf_model_name() else "ollama"

        if requested in ("", "ollama"):
            return "ollama"

        logger.warning("[%s] unknown engine='%s', using ollama", self.name, requested)
        return "ollama"

    @retry(max_attempts=3, delay=1.0)
    async def _call_llm(self, prompt: str) -> str:
        if self.engine == "huggingface":
            if self.hf_llm is None:
                raise ModelNotLoadedError(
                    f"[{self.name}] engine='huggingface' but hf_llm is not "
                    "initialized. Subclasses must set self.hf_llm in __init__.",
                    context={
                        "agent": self.name,
                        "engine": self.engine,
                        "hf_model_setting": self.hf_model_setting,
                    },
                )
            return await self.hf_llm.generate(
                prompt,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                seed=_DEFAULT_SEED,
            )

        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": self.temperature,
                "seed": _DEFAULT_SEED,
            },
        }
        client = get_ollama_client()
        response = await client.post(self.settings.ollama_url, json=payload)
        response.raise_for_status()
        data = response.json()
        return data.get("response", "").strip()
```

### Разбор ключевых решений

**Почему `BaseAgent(ABC)` и `Agent(BaseAgent)` — два класса, а не один.** `Validator` не вызывает LLM. Если бы он наследовался от `Agent`, у него были бы `_call_llm`, `_resolve_engine`, `hf_llm` — мёртвый код, который вводит в заблуждение («почему Validator не использует эти поля?»). Разделение: `BaseAgent` — контракт «агент», `Agent` — «агент, которому нужна LLM».

**Почему `hf_model_setting`/`ollama_model_setting` — class-level строки, а не аргументы `__init__`.** Имена полей `Settings` — константа класса. У `Critic` это `"HF_CRITIC_MODEL"`/`"CRITIC_MODEL"`, у `Editor` — `"HF_EDITOR_MODEL"`/`"EDITOR_MODEL"`. `_resolve_engine` использует `getattr(self.settings, self.hf_model_setting)` — динамический доступ по имени. Если бы передавали в `__init__`, пришлось бы дублировать в каждом подклассе.

**Почему `_resolve_engine` возвращает только `"ollama"` или `"huggingface"`, не `"auto"`.** `auto` — это **политика**, а не движок. Политика резолвится один раз в `__init__`: «если HF-модель задана — HF, иначе Ollama». Дальше `self.engine` — конкретное значение. Это упрощает `_call_llm` (нет `if engine == "auto"`).

**Почему `requested == "huggingface"` без HF-модели → warning + `"ollama"`.** Студент поставил `AGENT_ENGINE=huggingface`, но забыл `HF_CRITIC_MODEL`. Падать с `ModelNotLoadedError` при первом запросе — плохой UX. Warning + fallback на Ollama — лучше: система работает, студент видит в логах «engine=huggingface but HF_CRITIC_MODEL is empty». Это явное поведение.

**Почему `requested in ("", "ollama")` → `"ollama"`, а не `ValueError`.** Пустая строка — частый случай при незаданной переменной окружения. `""` трактуется как «не задано» → дефолт `ollama`. Валидатор в `Settings` (Этап 2) уже проверяет `AGENT_ENGINE in {auto, ollama, huggingface}`, так что `""` до `_resolve_engine` не дойдёт через `Settings.AGENT_ENGINE`, но может прийти из `Critic(engine="")`. Пустая строка → ollama — безопасный дефолт.

**Почему `_DEFAULT_SEED = 42` — module-level.** Одно значение на весь модуль. `_call_llm` использует его в двух местах (HF и Ollama). Если поменять в одном — забыть во втором, воспроизводимость сломается. Одна константа — одна точка правды.

**Почему seed = 42, а не случайный.** `42` — не магическое число, а просто фиксированное значение. Любое константное число подходит. `42` — распространённое соглашение. Важно, что оно **постоянно**: один вход → один выход.

**Почему `@retry(max_attempts=3, delay=1.0)` на `_call_llm`, а не на `generate`.** `_call_llm` — единственная точка вызова LLM в агенте. Обернуть `generate` — значит, что 3 попытки будут только на HF, а на Ollama — одна. Обернуть `_call_llm` — единая политика ретраев для обоих движков.

**Почему `raise ModelNotLoadedError` с полным контекстом.** `{"agent": name, "engine": engine, "hf_model_setting": setting}` — при падении в логе сразу видно, какой агент и какая настройка. Без контекста — просто «hf_llm is None», непонятно, где искать.

**Почему `get_ollama_client()` вызывается внутри `_call_llm`.** Функция привязана к текущему event loop через `WeakKeyDictionary`. Если бы клиент кэшировался в `__init__`, он был бы привязан к loop'у, в котором создан агент. В FastAPI агент живёт дольше одного запроса, но каждый запрос может попасть в тот же loop — ок. В Celery один loop на процесс — тоже ок. Но если агент создаётся в Celery worker с loop1, а используется в loop2 (не должно случаться, но защищаемся), `get_ollama_client()` вернёт правильный клиент для активного loop.

**Почему `data.get("response", "").strip()`.** Ollama `/api/generate` возвращает `{"response": "...", "done": true, ...}`. Если поле `response` пустое (модель не ответила) — вернуть `""`. `.strip()` — на случай пробелов/новых строк в начале/конце.

**Почему `response.raise_for_status()`.** `httpx` не бросает по умолчанию. Без `raise_for_status()` ответ 500 от Ollama выглядел бы как успешный с пустым `response`. С ним — `HTTPStatusError`, `@retry` её поймает, повторит.

**Почему `temperature: float = 0.0` на `Agent` по умолчанию.** Critic должен быть детерминированным (`0.0`). Editor нужен `0.3` (переопределяет в `__init__`). Дефолт `0.0` — безопасный: агент, который не переопределил, не будет случайным.

**Почему `max_new_tokens: int = 512`.** 512 токенов хватает на стандартный перевод одного сегмента (2–3 предложения). Для `Critic` (JSON с 7 полями) — тоже с запасом. Переопределяется в подклассе, если нужно.

### Проверка

```powershell
python -m poetry run python -c "from backend.agents.base import BaseAgent, Agent; print('BaseAgent abstract:', BaseAgent.__abstractmethods__); print('Agent abstract:', Agent.__abstractmethods__)"
```

**Что вы должны увидеть:**

```
BaseAgent abstract: frozenset({'process'})
Agent abstract: frozenset({'process'})
```

Оба класса абстрактны — `process` обязателен. Конкретный агент (`Critic`, `Editor`, `Validator`) его реализует.

### Коммит

```powershell
git add backend/agents/base.py
git commit -m "feat(week-5): add BaseAgent/Agent with engine resolution and seed=42"
```

---

## 5.6. `backend/agents/supervisor.py` — оркестратор

**Что это:** класс `Supervisor` с `process(state)` — гоняет `Validator → Critic → Editor → Validator → Critic → ...` с адаптивным бюджетом, rollback при регрессии, флагами `use_critic`/`use_editor`.

Создайте `backend/agents/supervisor.py` — полный код из референса:

```python
# backend/agents/supervisor.py
import logging
from typing import Any, Dict, Optional

from backend.core.config import get_settings
from backend.core.constants import (
    ADAPTIVE_BUDGET_LOW,
    ADAPTIVE_BUDGET_MID,
    CRITIC_PASS_THRESHOLD,
)

logger = logging.getLogger(__name__)


class Supervisor:
    """Orchestrates the agent pipeline."""

    _NON_SNAPSHOT_KEYS = frozenset({
        "reasoning_log",
        "editor_rolled_back",
        "iteration",
    })

    _EPHEMERAL_KEYS = frozenset({
        "critic_adequacy",
        "critic_fluency",
        "critic_terminology",
        "critic_score",
        "critic_reasoning",
        "critic_suggestions",
        "critic_missing_terms",
        "critic_failed",
        "critic_passed",
        "validator_errors",
        "validator_warnings",
        "validation_passed",
        "supervisor_iterations",
        "supervisor_final",
        "supervisor_stopped_reason",
        "supervisor_budget",
    })

    _EDITOR_DEFAULTS: Dict[str, Any] = {
        "editor_changed": False,
        "editor_failed": False,
        "editor_original": None,
        "editor_corrected": None,
        "editor_iterations": 0,
    }

    def __init__(self, max_iterations: int = 3, engine: Optional[str] = None):
        self.max_iterations = max_iterations
        self.engine = engine or get_settings().AGENT_ENGINE
        self._iteration = 0
        self._validator = None
        self._critic = None
        self._editor = None

    def _get_validator(self):
        if self._validator is None:
            from backend.agents.validator import Validator
            self._validator = Validator(engine=self.engine)
        return self._validator

    def _get_critic(self):
        if self._critic is None:
            from backend.agents.critic import Critic
            self._critic = Critic(engine=self.engine)
        return self._critic

    def _get_editor(self):
        if self._editor is None:
            from backend.agents.editor import Editor
            self._editor = Editor(engine=self.engine)
        return self._editor

    @staticmethod
    def _has_actionable_suggestions(state: Dict[str, Any]) -> bool:
        return bool(state.get("critic_suggestions") or [])

    @classmethod
    def _adaptive_budget(cls, state: Dict[str, Any]) -> int:
        score = state.get("critic_score")

        if score is None or score < ADAPTIVE_BUDGET_LOW:
            return 3
        if score < ADAPTIVE_BUDGET_MID:
            return 2

        if not state.get("validation_passed", True):
            return 1
        if cls._has_actionable_suggestions(state):
            return 1
        return 0

    @classmethod
    def _needs_correction(cls, state: Dict[str, Any]) -> bool:
        if state.get("critic_failed"):
            return False

        validation_passed = state.get("validation_passed", True)
        critic_score = state.get("critic_score")

        if critic_score is None:
            return not validation_passed

        if not validation_passed:
            return True
        if critic_score < CRITIC_PASS_THRESHOLD:
            return True
        return cls._has_actionable_suggestions(state)

    @staticmethod
    def _is_improvement(prev: Dict[str, Any], curr: Dict[str, Any]) -> bool:
        if curr.get("critic_failed"):
            return False

        prev_score = prev.get("critic_score")
        curr_score = curr.get("critic_score")

        if curr_score is None and prev_score is not None:
            return False
        if curr_score is not None and prev_score is None:
            return True
        if curr_score is not None and prev_score is not None:
            if curr_score > prev_score:
                return True
            if curr_score < prev_score:
                return False

        if curr.get("validation_passed") and not prev.get("validation_passed"):
            return True
        if prev.get("validation_passed") and not curr.get("validation_passed"):
            return False

        return False

    @staticmethod
    def _compute_final(state: Dict[str, Any]) -> bool:
        if state.get("critic_failed"):
            return False
        if not state.get("validation_passed", False):
            return False
        score = state.get("critic_score")
        if score is None:
            return True
        return score >= CRITIC_PASS_THRESHOLD

    @classmethod
    def _snapshot(cls, state: Dict[str, Any]) -> Dict[str, Any]:
        snap: Dict[str, Any] = {}
        for key, value in state.items():
            if key in cls._NON_SNAPSHOT_KEYS:
                continue
            snap[key] = list(value) if isinstance(value, list) else value
        return snap

    @classmethod
    def _restore(cls, state: Dict[str, Any], snapshot: Dict[str, Any]) -> None:
        for key in list(state.keys()):
            if key not in snapshot and key not in cls._NON_SNAPSHOT_KEYS:
                del state[key]
        for key, value in snapshot.items():
            state[key] = list(value) if isinstance(value, list) else value

    def _finalize(
        self,
        state: Dict[str, Any],
        reason: str,
        initial_score: Optional[float] = None,
        budget: int = 0,
    ) -> Dict[str, Any]:
        final_passed = self._compute_final(state)
        state["supervisor_iterations"] = self._iteration
        state["supervisor_final"] = final_passed
        state["supervisor_stopped_reason"] = reason
        state["supervisor_budget"] = budget
        state["reasoning_log"].append({
            "agent": "Supervisor",
            "step": "final",
            "iterations": self._iteration,
            "final_passed": final_passed,
            "stopped_reason": reason,
            "editor_rolled_back": state.get("editor_rolled_back", False),
            "budget": budget,
            "initial_score": initial_score,
            "final_score": state.get("critic_score"),
        })
        logger.info(
            "Final: iterations=%s, passed=%s, rolled_back=%s, "
            "budget=%s, reason=%s",
            self._iteration,
            final_passed,
            state.get("editor_rolled_back", False),
            budget,
            reason,
        )
        return state

    async def process(self, state: Dict[str, Any]) -> Dict[str, Any]:
        self._iteration = 0
        state["editor_rolled_back"] = False
        state["reasoning_log"] = []

        for key in self._EPHEMERAL_KEYS:
            state.pop(key, None)

        for key, value in self._EDITOR_DEFAULTS.items():
            state[key] = value

        use_critic = bool(state.get("use_critic", True))
        use_editor = bool(state.get("use_editor", True))

        validator = self._get_validator()
        state = await validator.process(state)

        if not use_critic and not use_editor:
            logger.info("Agents disabled, returning after validator")
            return self._finalize(state, "agents_disabled")

        if use_critic:
            critic = self._get_critic()
            state = await critic.process(state)

            if state.get("critic_failed"):
                logger.info("Stopping early: critic_failed")
                return self._finalize(state, "critic_failed")

        if not use_editor:
            logger.info("Editor disabled, returning after critic")
            return self._finalize(state, "editor_disabled")

        editor = self._get_editor()
        initial_score = state.get("critic_score")
        needs_correction = self._needs_correction(state)
        adaptive_budget = self._adaptive_budget(state) if needs_correction else 0
        effective_max = min(self.max_iterations, adaptive_budget)
        state["supervisor_budget"] = effective_max

        logger.info(
            "initial_score=%s, adaptive_budget=%s, effective_max=%s, "
            "needs_correction=%s",
            initial_score,
            adaptive_budget,
            effective_max,
            needs_correction,
        )

        snapshot = self._snapshot(state)

        if not needs_correction:
            stopped_reason = "converged"
        elif effective_max == 0:
            stopped_reason = "budget_exhausted"
        else:
            stopped_reason = "max_iterations"

        while needs_correction and self._iteration < effective_max:
            self._iteration += 1
            state["iteration"] = self._iteration

            state = await editor.process(state)
            editor_failed = state.get("editor_failed", False)
            editor_changed = state.get("editor_changed", False)

            state = await validator.process(state)
            if use_critic:
                state = await critic.process(state)

            if editor_failed or (use_critic and state.get("critic_failed")):
                logger.info(
                    "Rollback: editor_failed=%s, critic_failed=%s",
                    editor_failed,
                    state.get("critic_failed"),
                )
                self._restore(state, snapshot)
                state["editor_rolled_back"] = True
                stopped_reason = "rollback_error"
                break

            if not editor_changed:
                logger.info("Editor did not change translation, stopping.")
                state["editor_rolled_back"] = False
                stopped_reason = "no_change"
                break

            if not self._is_improvement(snapshot, state):
                logger.info(
                    "Rollback: no improvement (%s -> %s)",
                    snapshot.get("critic_score"),
                    state.get("critic_score"),
                )
                self._restore(state, snapshot)
                state["editor_rolled_back"] = True
                stopped_reason = "rollback_regression"
                break

            snapshot = self._snapshot(state)
            state["editor_rolled_back"] = False
            needs_correction = self._needs_correction(state)

            if not needs_correction:
                stopped_reason = "converged"

        return self._finalize(
            state,
            stopped_reason,
            initial_score=initial_score,
            budget=effective_max,
        )

    def get_status(self) -> Dict[str, Any]:
        critic = self._critic
        validator = self._validator
        return {
            "iterations": self._iteration,
            "validator_passed": (
                validator.get_last_result().get("passed", False)
                if validator and validator.get_last_result()
                else False
            ),
            "critic_score": (
                critic.get_last_result().get("overall_score", 0)
                if critic and critic.get_last_result()
                else 0
            ),
        }
```

### Разбор ключевых решений

**Почему `_NON_SNAPSHOT_KEYS` включает `reasoning_log`.** При rollback мы восстанавливаем state, но **не** хотим терять записи о провалившейся итерации — это аудит для отладки. `reasoning_log` продолжает накапливаться, даже если перевод откатился. `editor_rolled_back` — флаг, устанавливаемый «снаружи» после `_restore`, тоже не должен восстанавливаться (иначе `_restore` его затрёт). `iteration` — счётчик, управляется в `process`, не в snapshot.

**Почему `_EPHEMERAL_KEYS` — frozenset из 17 полей.** `process` в начале очищает всё, что относится к предыдущему прогону. Один `state`-dict можно переиспользовать: `/translate` может вызвать `supervisor.process` дважды с разным текстом, и если поля не сброшены, второй прогон начнётся с `critic_score=8.0` от первого и сразу `converged`. `frozenset` — константа, не список, чтобы никто случайно не сделал `append`.

**Почему `_EDITOR_DEFAULTS` — dict, а не поля класса.** `process` выставляет их в state перед прогоном. Без сброса `editor_changed=True` от прошлого прогона попадёт в следующий. `dict` — чтобы обходить через `.items()`.

**Почему `__init__` не создаёт агентов, а `_get_*` создаёт лениво.** `Supervisor(engine="ollama")` при импорте не должен дёргать `Critic.__init__`, который импортирует `get_hf_llm` (тяжёлый модуль с torch). Ленивая инициализация — `_get_critic()` вызывается только если `use_critic=True`. В `agents_only` режиме `Critic` и `Editor` создаются, в `baseline` — нет.

**Почему `_has_actionable_suggestions` — `bool(state.get("critic_suggestions") or [])`.** `critic_suggestions` может быть `None`, `[]`, или списком. `or []` нормализует `None` в `[]`, `bool(...)` — в `True`/`False`. Пустой список и `None` → `False`, непустой → `True`.

**Почему `_adaptive_budget` — `@classmethod`, а не `@staticmethod`.** Он вызывает `cls._has_actionable_suggestions` — тоже `classmethod`. Если оба класса наследуются (`SubSupervisor`), `cls` — подкласс, и вызовы резолвятся через него. Мелочь, но стандарт.

**Почему `_is_improvement` проверяет `critic_failed` первым.** Если после итерации Critic упал — это уже регрессия, независимо от score. В `process` это дублируется в `if editor_failed or critic_failed` (rollback_error), но `_is_improvement` дополнительно защищает от вызова в других контекстах (например, если кто-то использует `_is_improvement` напрямую).

**Почему `_is_improvement` при `curr_score is None` и `prev_score is not None` возвращает `False`.** Score был, стал `None` — регрессия (Critic сломался). Обратное (`prev=None, curr=score`) → `True`: «было нечего оценить, стало лучше».

**Почему `_is_improvement` в конце возвращает `False`, даже если score равны.** Равные score = не улучшение. Система должна остановиться (`no_change`), а не крутить Editor вхолостую. Это и есть смысл rollback: «попытка не улучшила — откат».

**Почему `_compute_final`: `score is None` → `True`.** Если Critic не вызывался (`baseline`, `glossary`, `tm`, `tm_hybrid`) — `critic_score=None`, но валидация прошла. Это валидный финальный результат. `True` — «проверки, которые были, прошли».

**Почему `_snapshot` — `@classmethod`, не `@staticmethod`.** Использует `cls._NON_SNAPSHOT_KEYS`. Если подкласс переопределит `_NON_SNAPSHOT_KEYS` — снимок учтёт это.

**Почему `_snapshot` копирует списки через `list(value)`.** `state` содержит `reasoning_log` (list), `critic_suggestions` (list), `glossary_used` (list объектов) и т.д. Shallow copy списка защищает от мутации: если после снимка Editor добавит элемент в `critic_suggestions`, snapshot не изменится. Вложенные объекты (dict'ы в reasoning_log) — не копируются, но они там не мутируются, так что shallow достаточно.

**Почему `_restore` сначала удаляет лишние ключи, потом восстанавливает.** Если итерация добавила `editor_iterations=1`, а в snapshot его не было (первая итерация), при rollback `editor_iterations` должен исчезнуть. Цикл `for key in list(state.keys())` + `del state[key]` — единственный способ. `list(state.keys())` нужен, потому что `del` во время итерации по `state.keys()` бросает `RuntimeError`.

**Почему `_finalize` всегда вызывается в конце `process`.** Единая точка: пишет `supervisor_iterations`, `supervisor_final`, `supervisor_stopped_reason`, `supervisor_budget`, добавляет запись в `reasoning_log`. Все `return` в `process` идут через `_finalize` — гарантия, что state всегда консистентен.

**Почему в `process` `state["reasoning_log"] = []` в начале, но `editor_rolled_back = False` тоже.** `reasoning_log` — должен быть пуст для нового прогона. `editor_rolled_back` — флаг конкретного прогона, если не сбросить, в `_finalize` попадёт `True` от прошлого.

**Порядок `use_critic`/`use_editor` в `process`:**

1. `Validator` — всегда.
2. Если **оба** `False` → `agents_disabled`, return.
3. Если `use_critic` → `Critic`, при `critic_failed` → `critic_failed`, return.
4. Если `use_editor` — False → `editor_disabled`, return.
5. Дальше цикл Editor.

**Почему `use_critic=False, use_editor=True` идёт в цикл без `initial_score`.** В этом режиме `critic_score=None` (Critic не вызывался). `_needs_correction` вернёт `not validation_passed`. `_adaptive_budget` вернёт `3` (score None). Значит, цикл сделает до 3 итераций, но `_is_improvement` будет `False` на первой же (нет score, нет изменений в `validation_passed` — если Editor не изменил, будет `no_change`; если изменил, но `validation_passed` осталось `False`, `_is_improvement` вернёт `False` → rollback). Это правильное поведение: без Critic сложно измерить «стало лучше».

**Почему `state["iteration"] = self._iteration` внутри цикла, а не в начале.** `iteration` в state — это то, что видит пользователь в ответе (в `reasoning_log` или в `TranslateResponse.iterations`). Если положить в начале, при `effective_max=0` будет `iteration=0` в state, но счётчик `self._iteration` тоже 0. Смысл `state["iteration"]` — «на какой итерации остановились», ставится на каждой итерации. При `converged` с 0 итераций — field не выставляется, и в ответе его нет (или `None`). Это ожидаемо.

**Почему `if not editor_changed: stopped_reason="no_change"; break` идёт **после** `_is_improvement`.** Порядок: сначала проверяем `editor_failed`/`critic_failed` (rollback_error), потом `editor_changed` (no_change — нормальная остановка, не регрессия), потом `_is_improvement` (rollback_regression). Если Editor изменил и score улучшился — продолжаем. Если не изменил — break без rollback (state не откатывается, потому что изменения не было). Если изменил, но score упал — rollback.

**Почему `snapshot = self._snapshot(state)` **в конце** каждой успешной итерации.** После итерации state обогатился новым score, новыми `critic_suggestions`. Для следующей итерации snapshot должен быть «состояние после последней успешной». Без этого повторный snapshot указывал бы на старое, и `_is_improvement(snapshot, state)` всегда был бы True.

**Почему `state["editor_rolled_back"] = False` после успешной итерации.** Если предыдущая итерация откатилась, а текущая — улучшила, флаг сбрасывается. В `_finalize` попадёт `False` — «последняя итерация была успешной».

### Эксперименты

**Эксперимент 1: `baseline` (оба флага False).**

```powershell
python -m poetry run python -c "
import asyncio
from backend.agents.supervisor import Supervisor
async def main():
    s = Supervisor()
    state = {
        'source_text': 'Hello', 'source_lang': 'en', 'target_lang': 'ru',
        'translation': 'Привет', 'glossary_used': [],
        'use_critic': False, 'use_editor': False,
    }
    r = await s.process(state)
    print('stopped:', r['supervisor_stopped_reason'])
    print('iterations:', r['supervisor_iterations'])
asyncio.run(main())
"
```

**Что вы должны увидеть:**

```
stopped: agents_disabled
iterations: 0
```

(Импорт `Validator` в `_get_validator` сработает, но `Validator.process` использует `embedding_model.encode` для семантического поиска — на Этапе 5 это требует загрузки `sentence-transformers`. Если модель не загружена — будет warning в логе, но проверки отработают.)

**Эксперимент 2: `use_critic=True, use_editor=False`.**

```powershell
python -m poetry run python -c "
import asyncio
from backend.agents.supervisor import Supervisor
async def main():
    s = Supervisor()
    state = {
        'source_text': 'Hello', 'source_lang': 'en', 'target_lang': 'ru',
        'translation': 'Привет',
        'use_critic': True, 'use_editor': False,
    }
    r = await s.process(state)
    print('stopped:', r['supervisor_stopped_reason'])
    print('critic_score:', r.get('critic_score'))
asyncio.run(main())
"
```

**Что вы должны увидеть** (Critic ещё с заглушкой — вернёт 5.0, что ниже 7.0, но `use_editor=False` → остановка):

```
stopped: editor_disabled
critic_score: 5.0
```

### Коммит

```powershell
git add backend/agents/supervisor.py
git commit -m "feat(week-5): add Supervisor with adaptive budget and rollback"
```

---

## 5.7. `backend/agents/validator.py` — rule-based валидатор

**Что это:** `Validator(BaseAgent)` — 7 проверок, LLM не вызывает. Принимает `engine` в `__init__` для единообразной сборки, но игнорирует.

Создайте `backend/agents/validator.py` — полный код из референса. Модуль содержит:
- константы `MIN_TERM_LEN=2`, `MIN_TERM_LEN_FOR_SEMANTIC=5`, `MIN_CONTENT_WORD_LEN=4`, `GLOSSARY_SEMANTIC_THRESHOLD=0.7`, `_MAX_LEFTOVER_REPORTED=5`, `_MIN_TARGET_SCRIPT_RATIO=0.5`;
- `_COMMON_STOPWORDS` — 150+ слов en+ru;
- `_SCRIPT_PATTERNS` — cyrillic/arabic/latin regex;
- `_LANG_SCRIPT` — маппинг 18 языков на скрипт;
- `_CJK_RE`, `_PLACEHOLDER_RE`, `_LETTER_RE`;
- `_contains_word`, `_extract_content_words`, `_glossary_pair`, `_semantic_match`;
- `Validator(BaseAgent)` с `process`, `_check_glossary`, `_check_echo`, `_check_placeholders`, `_check_script`, `_check_repetitions`, `_check_leftover_source`.

> **Объём файла** — ~400 строк. Полный код в референсном проекте (`backend/agents/validator.py`). Ниже — ключевые фрагменты и разбор.

```python
# backend/agents/validator.py (фрагмент структуры)
import json, logging, re
from typing import Any, Dict, List, Optional, Set

from backend.agents.base import BaseAgent
from backend.data.embeddings import embedding_model

logger = logging.getLogger(__name__)

MIN_TERM_LEN = 2
MIN_TERM_LEN_FOR_SEMANTIC = 5
MIN_CONTENT_WORD_LEN = 4
GLOSSARY_SEMANTIC_THRESHOLD = 0.7
_MAX_LEFTOVER_REPORTED = 5
_MIN_TARGET_SCRIPT_RATIO = 0.5

# ... _COMMON_STOPWORDS, _SCRIPT_PATTERNS, _LANG_SCRIPT, _CJK_RE,
#     _PLACEHOLDER_RE, _LETTER_RE

def _contains_word(text: str, term: str) -> bool:
    """Whole-word, case-insensitive match of ``term`` inside ``text``."""
    if not term or not text:
        return False
    pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"
    return re.search(pattern, text, re.IGNORECASE) is not None

def _extract_content_words(text: str) -> Set[str]:
    if not text:
        return set()
    words = re.findall(r"\w+", text.lower(), flags=re.UNICODE)
    return {
        w for w in words
        if len(w) >= MIN_CONTENT_WORD_LEN and w not in _COMMON_STOPWORDS
    }

def _glossary_pair(entry: Any) -> tuple[str, str]:
    if isinstance(entry, dict):
        source_term = entry.get("en") or ""
        target_term = entry.get("tgt") or ""
    else:
        source_term = getattr(entry, "en", "") or ""
        target_term = getattr(entry, "tgt", "") or ""
    return source_term.strip(), target_term.strip()

def _semantic_match(term: str, translation: str, threshold: float = GLOSSARY_SEMANTIC_THRESHOLD) -> bool:
    if not term or not translation:
        return False
    try:
        vectors = embedding_model.encode([term, translation])
        term_vec, trans_vec = vectors[0], vectors[1]
        cos = sum(a * b for a, b in zip(term_vec, trans_vec))
        return cos >= threshold
    except Exception as e:
        logger.warning("Semantic match failed for term '%s': %s", term, e)
        return False


class Validator(BaseAgent):
    """
    Rule-based validator. Does NOT call an LLM.
    """

    def __init__(self, engine: Optional[str] = None):
        super().__init__(name="Validator")

    async def process(self, state: Dict[str, Any]) -> Dict[str, Any]:
        logger.info("[%s] Starting...", self.name)

        source_text = state.get("source_text", "") or ""
        translation = state.get("translation", "") or ""
        target_lang = state.get("target_lang", "") or ""
        glossary_entries = state.get("glossary_used", []) or []

        errors: List[str] = []
        warnings: List[str] = []

        errors.extend(self._check_glossary(source_text, translation, glossary_entries))
        if not translation.strip():
            errors.append("Translation is empty")
        errors.extend(self._check_echo(source_text, translation))
        errors.extend(self._check_placeholders(source_text, translation))
        errors.extend(self._check_script(target_lang, translation))

        warnings.extend(self._check_repetitions(translation))
        warnings.extend(self._check_leftover_source(source_text, translation))

        state["validator_errors"] = errors
        state["validator_warnings"] = warnings
        state["validation_passed"] = len(errors) == 0

        self._last_result = {
            "passed": state["validation_passed"],
            "errors": errors,
            "warnings": warnings,
        }

        reasoning_entry = {
            "agent": self.name,
            "step": "validation",
            "errors": errors,
            "warnings": warnings,
            "passed": state["validation_passed"],
        }
        state.setdefault("reasoning_log", []).append(reasoning_entry)
        return state

    # ... _check_glossary, _check_echo, _check_placeholders,
    #     _check_script, _check_repetitions, _check_leftover_source
```

### Разбор ключевых решений

**Почему `_COMMON_STOPWORDS` — большой hardcoded set, а не `nltk.corpus.stopwords`.** `nltk` требует скачивания корпуса (`nltk.download("stopwords")`) при первом запуске — в Docker это дополнительный шаг. Плюс `nltk` не покрывает татарские/башкирские/таджикские стоп-слова. Проект поддерживает 18 языков, а `_COMMON_STOPWORDS` содержит базовый набор для en+ru, который **работает для всех языков** постольку-поскольку (проверка `_check_leftover_source` — warning, а не error, ложное срабатывание не критично).

**Почему `_contains_word` через `(?<!\w)...(?!\w)`, а не `\b...\b`.** `\b` — граница слова по `\w`, и в Unicode работает для кириллицы, но есть тонкости с дефисами. `(?<!\w)` / `(?!\w)` — явные lookbehind/lookahead, гарантируют «до и после терма нет `\w`». Для `"book"` в `"bookshelf"` — не сработает, что правильно. Для `"book"` в `"книга (book)"` — сработает.

**Почему `_glossary_pair` толерантна к dict и объекту.** `glossary_used` может содержать `GlossaryEntry` (Pydantic) или `dict` (если state прошёл через сериализацию, например в Celery). Без толерантности валидатор падал бы на Celery-пути.

**Почему `_semantic_match` вызывается только для терминов ≥ 5 символов.** `embedding_model.encode` — тяжёлая операция (~50 мс на пару). Для коротких терминов (`"the"`, `"is"`) буквальное вхождение в перевод почти гарантировано, а семантическое совпадение — случайно (cosine для двух коротких строк близок к 0.9 из-за общих служебных слов). Ограничение `≥ 5` отсекает эти ложные срабатывания.

**Почему `_check_glossary` при пустом `target_term` (len < 2) — skip, а не error.** Глоссарий может содержать короткие артикли (`a`, `и`). Их сложно проверить надёжно; пропуск — безопаснее, чем ложная ошибка.

**Почему `_check_glossary` при найденном `source_term` в источнике проверяет `_contains_word(translation, target_term)` буквально, потом семантически.** Буквальное — быстро (regex), семантическое — медленно (embedding). Сначала дешёвая проверка, при неудаче — дорогая. Типичный паттерн.

**Почему `_check_echo` нормализует `lower().split()` и `" ".join`.** `"Hello"` vs `"hello"` — эхо по смыслу. `"Hello  World"` vs `"hello world"` (двойной пробел) — тоже эхо. Нормализация через `" ".join(text.lower().split())` схлопывает все пробелы и приводит к нижнему регистру. Единственный способ поймать эхо с точностью.

**Почему `_check_placeholders` использует `_PLACEHOLDER_RE = r"%\d+\$[sd]|%[sd]"`.** Это два формата:
- `%s`, `%d` — positional printf (C-style).
- `%1$s` — indexed printf (позиционный).

Оба должны сохраняться при переводе. Если источник содержит `%s` (одно место для подстановки), а перевод — `%s %s`, это ошибка: при форматировании будет `TypeError`. Multiset-сравнение (`sorted`) ловит такие случаи.

**Почему `_check_script` использует `_MIN_TARGET_SCRIPT_RATIO = 0.5`, а не 1.0.** Идеальный перевод — 100% букв в целевом скрипте. Но имена (`"NASA"`, `"COVID-19"`) и аббревиатуры часто остаются в латинице даже в русском/татарском переводе. Порог 50% отсекает «перевод целиком на латинице» (ошибка), но пропускает «перевод с 2-3 латинскими словами» (нормально).

**Почему `_check_script` — error (не warning), а `_check_repetitions` — warning.** Скрипт-микс — критическая ошибка: `"Привет world"` для `target_lang="ru"` означает, что перевод сломан. Повтор слова — косметика: `"и и"` неприятно, но не ломает смысл. Разные уровни — разная серьёзность.

**Почему `_check_script` возвращает error при обнаружении CJK, даже если скрипт целевого языка — latin.** CJK-символы в переводе на английский — грубая ошибка (модель выдала китайский). Это гарантированно неправильно, error обоснован.

**Почему `_check_repetitions` проверяет только **последовательные** одинаковые слова.** `"и и"`, `"the the"` — очевидный баг перевода. `"и ... и"` через 5 слов — нормальная конструкция. Проверка только на позиции `i` и `i+1` — достаточна, ложных срабатываний нет.

**Почему `_check_leftover_source` — warning, не error.** Content-слово из источника в переводе — почти всегда плохо, но бывают легитимные случаи: имена собственные (`"Москва"` в английском тексте), термины (`"B-tree"` в русском тексте), аббревиатуры (`"JWT"`). Warning — правильно, error — нет.

**Почему `_check_leftover_source` останавливается на 5 словах.** Если перевод почти полностью на языке источника (100 content-слов), сообщение `"Possible leftover source words: [100 items]"` — неинформативно. Первые 5 — достаточно, чтобы понять проблему, остальное — шум.

**Почему `_check_glossary` использует `len(source_term) < MIN_TERM_LEN` (2) для пропуска, но `len(target_term) >= MIN_TERM_LEN_FOR_SEMANTIC` (5) для семантической проверки.** Разные пороги для разных целей: 2 — минимальная длина, чтобы вообще проверять (односимвольные термины — шум); 5 — минимальная длина, чтобы embedding-проверка была надёжной.

**Почему `Validator.__init__(engine=None)` принимает `engine`, но не использует.** `Supervisor._get_validator()` вызывает `Validator(engine=self.engine)`. Если бы `Validator.__init__` не принимал `engine`, `Supervisor` пришлось бы знать, что `Validator` — особый случай. Единый интерфейс (`__init__(engine=...)` у всех агентов) — проще. Комментарий в коде фиксирует: «engine accepted for uniform construction, not stored or used».

### Коммит

```powershell
git add backend/agents/validator.py
git commit -m "feat(week-5): add rule-based Validator (7 checks)"
```

---

## 5.8. `backend/agents/critic.py` — LLM-as-a-Judge

**Что это:** `Critic(Agent)` — оценивает перевод по 4 критериям (adequacy, fluency, terminology, overall_score) + reasoning + suggestions + missing_terms. Использует трёхуровневый few-shot (universal + domain + language) и сканер JSON для парсинга.

Создайте `backend/agents/critic.py` — полный код из референса. Модуль содержит:

- `_EXPECTED_KEYS = frozenset({"adequacy", "fluency", "terminology", "overall_score"})`;
- `_BULLET_PREFIX_RE = r"^\s*(?:[-\*•]|\d+[\.\)])(?:\s+|$)"`;
- `_COMMON_RULES` — 7 правил (кальки → fluency ≤ 5, терминология, максимум 8 без идеала, разные оценки при разнице, overall ≤ 8 при сомнении, fluency ≤ 5 при ошибке, fluency = 0 при утечке/CJK/meta);
- `_DOMAIN_RULES = {"medical": "8. ..."}`;
- `_LANGUAGE_RULES` — 16 языков (`ru`, `tt`, `ba`, `tg`, `fa`, `tr`, `kk`, `ky`, `udm`, `os`, `av`, `ar`, `bg`, `id`, `ms`, `uz`);
- `_UNIVERSAL_FEWSHOT` — 4 примера;
- `_DOMAIN_FEWSHOT = {}`;
- `_LANGUAGE_FEWSHOT` — 11 языков;
- `_PROMPT_TEMPLATE`;
- `Critic(Agent)` с `process`, `_get_fallback_evaluation`, `_build_prompt`, `_build_few_shot`, `_build_scoring_rules`, `_describe_active_tiers`, `_extract_json_spans`, `_parse_json_response`, `_format_glossary`, `_fallback_extract`, `_extract_list_block`.

> **Объём файла** — ~400 строк. Полный код в референсном проекте. Ниже — ключевые фрагменты (каркас `Critic` + `_extract_json_spans`) и разбор. Промпт-константы (`_COMMON_RULES`, `_LANGUAGE_RULES`, `_UNIVERSAL_FEWSHOT`, `_LANGUAGE_FEWSHOT`) копируйте из референсного кода — там 600+ строк текста, который не имеет смысла дублировать в методичке.

### Каркас `Critic` (полный, копируйте как есть)

```python
class Critic(Agent):
    hf_model_setting = "HF_CRITIC_MODEL"
    ollama_model_setting = "CRITIC_MODEL"

    def __init__(self, model_name: Optional[str] = None, engine: Optional[str] = None):
        super().__init__(name="Critic", engine=engine)

        if self.engine == "huggingface":
            self.model_name = model_name or self._hf_model_name()
            self.hf_llm = get_hf_llm(self.model_name)
        else:
            self.model_name = (
                model_name or self._ollama_model_name() or self.settings.LLM_MODEL
            ).replace("ollama/", "")
            self.hf_llm = None

        self.temperature = 0.0

    async def process(self, state: Dict[str, Any]) -> Dict[str, Any]:
        logger.info("[%s] Starting...", self.name)

        source_text = state.get("source_text", "")
        translation = state.get("translation", "")
        source_lang = state.get("source_lang", "en")
        target_lang = state.get("target_lang", "ru")
        glossary_entries = state.get("glossary_used", []) or []
        reference = state.get("reference")
        domain = state.get("domain", "general")
        audience = state.get("audience", "general")

        prompt = self._build_prompt(
            source=source_text, translation=translation,
            source_lang=source_lang, target_lang=target_lang,
            glossary_entries=glossary_entries, reference=reference,
            domain=domain, audience=audience,
        )

        failed = False
        try:
            response = await self._call_llm(prompt)
            evaluation = self._parse_json_response(response)
        except httpx.TimeoutException:
            logger.error("[%s] LLM timeout", self.name)
            evaluation = self._get_fallback_evaluation("Timeout error")
            failed = True
        except httpx.HTTPStatusError as e:
            logger.error("[%s] LLM HTTP error: %s", self.name, e.response.status_code)
            evaluation = self._get_fallback_evaluation(
                f"HTTP error: {e.response.status_code}"
            )
            failed = True
        except Exception as e:
            logger.error("[%s] Unexpected error: %s", self.name, e, exc_info=True)
            evaluation = self._get_fallback_evaluation("Unexpected error")
            failed = True

        if evaluation.get("failed"):
            failed = True

        suggestions = evaluation.get("suggestions") or []
        missing_terms = evaluation.get("missing_terms") or []
        overall_score = evaluation.get("overall_score")

        state["critic_adequacy"] = evaluation.get("adequacy")
        state["critic_fluency"] = evaluation.get("fluency")
        state["critic_terminology"] = evaluation.get("terminology")
        state["critic_score"] = overall_score
        state["critic_reasoning"] = evaluation.get("reasoning")
        state["critic_suggestions"] = suggestions
        state["critic_missing_terms"] = missing_terms
        state["critic_failed"] = failed
        state["critic_passed"] = (
            overall_score is not None
            and overall_score >= CRITIC_PASS_THRESHOLD
            and not failed
        )

        self._last_result = {
            "adequacy": evaluation.get("adequacy"),
            "fluency": evaluation.get("fluency"),
            "terminology": evaluation.get("terminology"),
            "overall_score": overall_score,
            "reasoning": evaluation.get("reasoning"),
            "suggestions": suggestions,
            "missing_terms": missing_terms,
            "failed": failed,
        }

        state.setdefault("reasoning_log", []).append(self._last_result | {
            "agent": self.name, "step": "critique",
        })

        return state

    def _get_fallback_evaluation(self, reason: str) -> Dict[str, Any]:
        return {
            "adequacy": None, "fluency": None, "terminology": None,
            "overall_score": None,
            "reasoning": f"Evaluation failed: {reason}",
            "suggestions": [], "missing_terms": [],
            "failed": True,
        }

    # ... _build_prompt, _build_few_shot, _build_scoring_rules,
    #     _describe_active_tiers, _extract_json_spans,
    #     _parse_json_response, _format_glossary, _fallback_extract,
    #     _extract_list_block
```

### `_extract_json_spans` — сканер с балансировкой скобок

```python
@staticmethod
def _extract_json_spans(text: str) -> List[str]:
    spans: List[str] = []
    i = 0
    n = len(text)
    while i < n:
        if text[i] != "{":
            i += 1
            continue
        depth = 0
        in_string = False
        escaped = False
        j = i
        while j < n:
            ch = text[j]
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = not in_string
            elif not in_string:
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        spans.append(text[i:j + 1])
                        break
            j += 1
        i = j + 1
    return spans
```

**Почему сканер, а не `re.search(r"\{.*?\}", ...)`.** Regex `.*?` не учитывает вложенность и строки. В ответе Critic ключ `reasoning` содержит текст перевода — если в нём есть `{` или `}`, regex обрежет JSON. `_extract_json_spans`:
- ведёт `depth` (вложенные объекты);
- отслеживает `in_string` (кавычки внутри значений);
- отслеживает `escaped` (обратный слэш перед кавычкой).

Сканер извлекает каждый `{...}` с балансировкой, включая вложенные. `json.loads` потом парсит.

### `_parse_json_response` — порядок кандидатов

```python
def _parse_json_response(self, response: str) -> Dict[str, Any]:
    candidates = self._extract_json_spans(response)
    if not candidates:
        logger.error("No JSON object found in response")
        return self._fallback_extract(response)

    parsed_candidates: List[Dict[str, Any]] = []
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if not isinstance(parsed, dict):
            continue
        parsed.pop("failed", None)
        parsed_candidates.append(parsed)

    if not parsed_candidates:
        logger.error("All JSON candidates failed to parse")
        return self._fallback_extract(response)

    for parsed in parsed_candidates:
        if _EXPECTED_KEYS.issubset(parsed.keys()):
            return parsed

    logger.warning("No candidate had all expected keys; using first parsed object")
    return parsed_candidates[0]
```

**Почему `parsed.pop("failed", None)`.** LLM может вернуть `{"failed": true, ...}` (гомоглиф нашего внутреннего флага). Если оставить, `evaluation.get("failed")` в `process` вернёт `True`, и Critic пометится как failed, даже если JSON валиден. Удаление `failed` из кандидата — защита от этого.

**Почему проверка `_EXPECTED_KEYS.issubset(parsed.keys())`.** LLM может вернуть несколько JSON-объектов в ответе (например, пример из few-shot и реальный ответ). Правильный — тот, у которого есть все 4 ключа. Первый такой — возвращаем.

**Почему при отсутствии полного — первый распарсенный.** Fallback: если ни один не полный, но хотя бы один валидный JSON — лучше использовать его (частичные данные) с warning, чем бросать `_fallback_extract` (regex, который тоже может не найти).

### Разбор ключевых решений

**Почему `temperature = 0.0`.** Critic должен быть детерминированным. Один и тот же перевод должен получать один и тот же score. Без `0.0` eval-режимы невоспроизводимы.

**Почему fallback возвращает `None`, а не 5.0.** Это критично. Если бы fallback был 5.0:
- Supervisor увидел бы `critic_score=5.0 < 7.0` → `_needs_correction=True`;
- Editor начал бы «исправлять» перевод на основе фейковой оценки;
- rollback-логика сломалась бы (сравнение `5.0` и реального score не имеет смысла).

С `None` + `failed=True`: Supervisor видит `critic_failed=True` в `process` и **останавливает пайплайн досрочно** с `stopped_reason="critic_failed"`. Это правильно: без работающего Critic дальнейший pipeline не имеет смысла.

**Почему `state["critic_passed"] = overall_score is not None and overall_score >= 7.0 and not failed`.** Три условия:
- score есть (не `None`);
- score ≥ порог;
- Critic не упал (`failed=False`).

Если Critic «упал, но вернул score» (что невозможно через `_get_fallback_evaluation`, но возможно через `_fallback_extract` с regex-парсингом) — `failed=True`, `passed=False`. Логично.

**Почему `_describe_active_tiers` — только для debug-лога.** Он показывает, какие тиры активированы: `common, domain=medical, language=ru`. Полезно при отладке, когда непонятно, почему Critic не применил правило про татарский. Не влияет на логику.

**Почему `_format_glossary` tolerant к dict/object.** Та же причина, что в Validator: `glossary_used` приходит из state, может быть `List[GlossaryEntry]` или `List[dict]`.

**Почему `_fallback_extract` использует regex `SCORE:\s*([\d.]+)`.** Если LLM ответил не JSON, а `SCORE: 7\nADEQUACY: 8\n...`, regex вытащит числа. Это последний шанс — если LLM совсем не в формате, `failed=True`, пайплайн останавливается.

**Почему `_extract_list_block` использует `_BULLET_PREFIX_RE` с обязательным пробелом после маркера.** Паттерн `r"^\s*(?:[-\*•]|\d+[\.\)])(?:\s+|$)"` — маркер `-`, `*`, `•` или `N.`/`N)`, затем **обязательно** пробел или конец строки. Без этого `"10.5 mg"` (в медицине) воспринялось бы как элемент списка, и `_extract_list_block` вернул бы `["5 mg"]` вместо `["10.5 mg"]`. Пробел после маркера — защита от десятичных дробей.

**Почему в `process` `if evaluation.get("failed"): failed = True`.** Двойная проверка. Первая — из `except`-веток (`failed=True` вручную). Вторая — если `evaluation` содержит `failed` (через `_fallback_extract`). Защищает от случая, когда fallback-экстрактор выставил `failed=True`, но код не в `except`-ветке.

**Почему `state["critic_failed"] = failed`, а не `state.setdefault(...)`.** Один раз на прогон, перезаписываем. Supervisor потом читает `state["critic_failed"]` напрямую.

### Временно: заглушка `_call_llm` в `Critic` на Этапе 5

До Этапа 8 `base.Agent._call_llm` реально ходит в Ollama. Ollama ещё не подключена. Значит, нужно **временно** переопределить `_call_llm` в `Critic`:

```python
# ВРЕМЕННО (убрать в Этапе 8): заглушка, возвращающая валидный JSON
async def _call_llm(self, prompt: str) -> str:
    return '{"adequacy": 5.0, "fluency": 5.0, "terminology": 5.0, "overall_score": 5.0, "reasoning": "Stub LLM: real call in Этап 8.", "suggestions": [], "missing_terms": []}'
```

**Зачем заглушка.** На Этапе 5 мы тестируем логику Critic (парсинг JSON, fallback, few-shot-композиция) без реальной Ollama. В Этапе 8 заглушка удаляется, `_call_llm` берётся из `Agent` — реальный вызов.

**Что тестировать с заглушкой.** `_extract_json_spans`, `_parse_json_response`, `_build_prompt` (few-shot), `_get_fallback_evaluation`. Всё, кроме самого HTTP-вызова. В Этапе 8 добавляется интеграционный тест с мок-Ollama.

### Коммит

```powershell
git add backend/agents/critic.py
git commit -m "feat(week-5): add Critic with 3-tier few-shot and JSON scanner (stub LLM)"
```

---

## 5.9. `backend/agents/editor.py` — LLM-редактор

**Что это:** `Editor(Agent)` — принимает текущий перевод + errors + warnings + suggestions, просит LLM исправить. Пропускает работу, если `critic_failed` или нет issues.

Полный код из референса (~200 строк, без больших промпт-констант). Структура:

```python
class Editor(Agent):
    hf_model_setting = "HF_EDITOR_MODEL"
    ollama_model_setting = "EDITOR_MODEL"

    def __init__(self, model_name: Optional[str] = None, engine: Optional[str] = None):
        super().__init__(name="Editor", engine=engine)
        if self.engine == "huggingface":
            self.model_name = model_name or self._hf_model_name()
            self.hf_llm = get_hf_llm(self.model_name)
        else:
            self.model_name = (
                model_name or self._ollama_model_name() or self.settings.LLM_MODEL
            ).replace("ollama/", "")
            self.hf_llm = None
        self.temperature = 0.3

    async def process(self, state: Dict[str, Any]) -> Dict[str, Any]:
        logger.info("[%s] Starting...", self.name)

        if state.get("critic_failed"):
            logger.info("[%s] Critic failed, skipping edit.", self.name)
            state["editor_changed"] = False
            state["editor_failed"] = False
            state.setdefault("reasoning_log", []).append({
                "agent": self.name, "step": "editing", "changed": False,
                "note": "Skipped: critic_failed",
            })
            return state

        translation = state.get("translation", "")
        errors = state.get("validator_errors", []) or []
        warnings = state.get("validator_warnings", []) or []
        suggestions = state.get("critic_suggestions", []) or []
        source_text = state.get("source_text", "")
        source_lang = state.get("source_lang", "en")
        target_lang = state.get("target_lang", "ru")

        if not errors and not warnings and not suggestions:
            state["editor_changed"] = False
            state["editor_failed"] = False
            state.setdefault("reasoning_log", []).append({
                "agent": self.name, "step": "editing", "changed": False,
                "note": "No issues to fix",
            })
            return state

        prompt = self._build_prompt(
            source_text=source_text, translation=translation,
            source_lang=source_lang, target_lang=target_lang,
            errors=errors, warnings=warnings, suggestions=suggestions,
        )

        editor_failed = False
        try:
            raw_corrected = await self._call_llm(prompt)
            corrected = clean_translation(raw_corrected)
        except Exception as e:
            logger.error("[%s] LLM call failed: %s", self.name, e, exc_info=True)
            corrected = translation
            editor_failed = True

        if not corrected or not corrected.strip():
            logger.warning("[%s] Empty correction, keeping original.", self.name)
            corrected = translation
            editor_failed = True

        changed = corrected != translation

        state["editor_failed"] = editor_failed
        state["editor_changed"] = changed
        state["editor_original"] = translation
        state["editor_corrected"] = corrected
        state["editor_iterations"] = state.get("editor_iterations", 0) + 1

        if changed:
            state["translation"] = corrected

        self._last_result = {
            "changed": changed, "failed": editor_failed,
            "original": translation, "corrected": corrected,
        }

        reasoning_entry = {
            "agent": self.name, "step": "editing",
            "original": translation, "corrected": corrected,
            "changed": changed, "failed": editor_failed,
        }
        state.setdefault("reasoning_log", []).append(reasoning_entry)

        return state

    @staticmethod
    def _build_prompt(...): ...
    @staticmethod
    def _format_issues(...): ...
```

### Разбор ключевых решений

**Почему `temperature = 0.3`, а не 0.0.** Editor — не детерминированный оценщик, а «творец». `0.0` часто даёт «безопасный» вариант — тот же перевод без изменений. `0.3` — баланс: достаточно вариативности, чтобы Editor рискнул что-то поменять, но не настолько, чтобы он «галлюцинировал» (при 0.7+ начинаются случайные перестановки). `seed=42` (из `Agent._call_llm`) делает результат воспроизводимым даже при `0.3`.

**Почему пропуск при `critic_failed`.** Если Critic упал, у нас нет `critic_suggestions`. Может быть, `validator_errors` есть — но Editor исправляет на основе **suggestions** от Critic. Без них он начнёт «улучшать» наугад, что чаще портит, чем помогает. Skip — правильно.

**Почему пропуск при отсутствии errors+warnings+suggestions.** Если всё пусто, что исправлять? Промпт «исправь ничего» даст либо тот же текст (пустая трата времени и токенов), либо случайные изменения (ухудшение). Skip — правильно.

**Почему `state["editor_iterations"] += 1` даже при skip.** Это счётчик «сколько раз Editor пытался». При skip — не увеличивается (в нашем коде). В `Supervisor._EDITOR_DEFAULTS` — 0. В `process` при skip `editor_iterations` не трогается, остаётся 0. Это ожидаемо.

**Почему `if not corrected or not corrected.strip() → editor_failed=True`.** LLM вернул `""` — либо пустой ответ, либо `clean_translation` всё вычистил. В любом случае это фейл (нет результата). Сохраняем оригинал (лучше ничего, чем пустая строка), `editor_failed=True` → Supervisor откатит.

**Почему `changed = corrected != translation` — строгое сравнение.** Пробелы в конце, разные регистры, пунктуация — всё считается изменением. Это правильно: если Editor вернул `"Привет "` (с пробелом), это изменение, и Supervisor должен его оценить. Иначе `no_change` срабатывал бы на «изменил пробел» и останавливал цикл зря.

**Почему `if changed: state["translation"] = corrected`.** Если не изменилось — не трогаем. Это защищает от случая, когда `corrected` совпадает с `translation`, но заменён другим объектом строки (не важно, но чище).

**Почему `except Exception` вокруг `_call_llm`, а не только `httpx`.** `_call_llm` бросает `ModelNotLoadedError`, `HTTPStatusError`, `TimeoutException`, `ValueError` (из `_encode_password`, если вдруг), `KeyError` (если `data` без `response`). Все ведут к одному: Editor не смог исправить → сохраняем оригинал, `editor_failed=True`. `except Exception` — правильно.

**Почему `raw_corrected` от `_call_llm` идёт через `clean_translation`.** LLM может вернуть `{"translation": "привет"}` или ` ```json... ``` ` или `Translation: привет`. `clean_translation` нормализует. Без этого Editor «изменил» бы перевод на JSON-обёртку — регрессия, rollback.

**Почему промпт содержит «NO-CHANGE is better than a broken change».** Это ключевой приём: LLM склонны «улучшать» даже когда не просят. Явное разрешение не меняться снижает частоту «сломанных правок», которые потом откатываются. Снижает число rollback'ов.

**Почему промпт содержит примеры Incorrect Edits.** Правило «применяй suggestion буквально» — абстрактно. Пример C (добавил `у него` — сломал грамматику) — конкретно. Few-shot с контрпримерами работает лучше, чем без них.

**Почему `_format_issues` собирает errors+warnings+suggestions в один текст.** Промпт — единый блок «Issues to fix». Разделение на три секции внутри блока — для читаемости LLM. Один блок — проще структурировать `_build_prompt`.

### Временно: заглушка `_call_llm` в `Editor` на Этапе 5

```python
# ВРЕМЕННО (убрать в Этапе 8): заглушка
async def _call_llm(self, prompt: str) -> str:
    return "Заглушка: перевод не изменён"
```

**Зачем заглушка.** LLM ещё не подключена. Тестируем логику Editor (skip при `critic_failed`, skip при отсутствии issues, обработка пустого ответа). В Этапе 8 — реальный вызов.

### Коммит

```powershell
git add backend/agents/editor.py
git commit -m "feat(week-5): add Editor with skip logic (stub LLM)"
```

---

## 5.10. Тесты

Тесты скинет пользователь отдельно. Здесь — **что** должно проверяться.

### `tests/unit/test_base_agent.py`

- `BaseAgent.__abstractmethods__` содержит `process`.
- `Agent._resolve_engine("auto")` при заполненном `HF_CRITIC_MODEL` → `"huggingface"`; при пустом → `"ollama"`.
- `Agent._resolve_engine("huggingface")` при пустом HF → `"ollama"` (fallback).
- `Agent._resolve_engine("ollama")` → `"ollama"`.
- `Agent._resolve_engine("google")` (неизвестный) → `"ollama"` с warning.
- `_call_llm` пиннит `seed=42` (проверить через mock `get_ollama_client`).
- `ModelNotLoadedError` при `engine="huggingface"` и `hf_llm=None`.

### `tests/unit/test_supervisor.py`

Восемь сценариев из критериев + ещё:
- `agents_disabled`: `use_critic=False, use_editor=False` → один `Validator.process`, `stopped_reason="agents_disabled"`.
- `critic_failed`: mock Critic возвращает `failed=True` → ранний выход, `stopped_reason="critic_failed"`.
- `editor_disabled`: `use_critic=True, use_editor=False` → после Critic, `stopped_reason="editor_disabled"`.
- `converged`: score=8.0, всё чисто → 0 итераций, `stopped_reason="converged"`.
- `budget_exhausted`: `use_editor=True, score=8.0, validation_passed=False` → `effective_max=1`, отработал одну итерацию.
- `rollback_regression`: mock Critic возвращает 8.0 → 5.0 → state восстановлен, `editor_rolled_back=True`, `stopped_reason="rollback_regression"`.
- `rollback_error`: mock Editor `editor_failed=True` → rollback, `stopped_reason="rollback_error"`.
- `no_change`: mock Editor `editor_changed=False` → break, `stopped_reason="no_change"`.
- `_snapshot`/`_restore`: список в snapshot, мутация после, restore возвращает старое.

**Как мокать агентов.** `Supervisor._get_validator/_get_critic/_get_editor` — импортируют модули lazily. В тесте: `monkeypatch.setattr(supervisor, "_get_critic", lambda: FakeCritic())` или `monkeypatch.setattr("backend.agents.critic.Critic", FakeCritic)`. Второй вариант чище — заменяет класс для всех экземпляров.

### `tests/unit/test_validator.py`

- `_check_glossary`: термин есть в источнике, отсутствует в переводе → error.
- `_check_glossary`: термин есть буквально в переводе → no error.
- `_check_glossary`: термин есть семантически (mock `_semantic_match` → True) → no error.
- `_check_glossary`: короткий термин (< 2) → skip.
- `_check_echo`: identical → error.
- `_check_echo`: разные регистры, разные пробелы → error.
- `_check_placeholders`: `%s` → `%s` → ok.
- `_check_placeholders`: `%s` → `%s %s` → error.
- `_check_script`: target="ru", translation="Привет" → ok.
- `_check_script`: target="ru", translation="Hello" → error.
- `_check_script`: target="ru", translation="Привет world" (50% латиницы) → пограничный случай (проверить ratio).
- `_check_script`: target="ru", translation содержит CJK → error.
- `_check_repetitions`: `"и и"` → warning.
- `_check_repetitions`: `"и в"` → no warning.
- `_check_leftover_source`: content слово из источника в переводе → warning.
- `_check_leftover_source`: стоп-слово → no warning.
- `_check_leftover_source`: > 5 → обрезано до 5.

### `tests/unit/test_critic.py`

- `_extract_json_spans`: простой `{...}` → 1 span.
- `_extract_json_spans`: `{a: {b: 1}}` → 1 span (с вложенностью).
- `_extract_json_spans`: `{a} {b}` → 2 span.
- `_extract_json_spans`: `{"reasoning": "with {braces}"}` → 1 span (строки учтены).
- `_extract_json_spans`: `{"reasoning": "with \"escaped\" quotes"}` → 1 span (escape учтён).
- `_parse_json_response`: валидный JSON с 4 ключами → dict.
- `_parse_json_response`: ` ```json\n{...}\n``` ` → dict.
- `_parse_json_response`: `Some text {valid json} more text` → dict.
- `_parse_json_response`: невалидный → `_fallback_extract`.
- `_parse_json_response`: JSON без `terminology` → warning + первый объект.
- `_parse_json_response`: `{"failed": true, ...}` → `failed` удалён.
- `_fallback_extract`: текст с `SCORE: 7\nADEQUACY: 8` → `overall_score=7, adequacy=8`.
- `_get_fallback_evaluation`: все поля `None`, `failed=True`.
- `process` с mock `_call_llm` → `TimeoutException` → `critic_failed=True`, `critic_score=None`.
- `_build_few_shot("ru", "general")` содержит `_UNIVERSAL_FEWSHOT` + `_LANGUAGE_FEWSHOT["ru"]`.
- `_build_scoring_rules("medical", "general")` содержит `_COMMON_RULES` + `_DOMAIN_RULES["medical"]`.
- `_build_prompt` содержит `target_lang` и `source` в правильных местах.

**Mock `_call_llm`.** `monkeypatch.setattr(Critic, "_call_llm", AsyncMock(return_value='{"adequacy":...}'))`. `AsyncMock` из `unittest.mock` — потому что `_call_llm` async.

### `tests/unit/test_editor.py`

- `critic_failed=True` → skip, `editor_changed=False`, `note="Skipped: critic_failed"`.
- нет errors/warnings/suggestions → skip, `note="No issues to fix"`.
- есть ошибки + LLM вернул изменённый текст → `editor_changed=True`, `translation` обновлён.
- есть ошибки + LLM вернул пустой ответ → `editor_failed=True`, translation не изменён.
- есть ошибки + LLM бросил `TimeoutException` → `editor_failed=True`, translation не изменён.
- есть ошибки + LLM вернул идентичный текст → `editor_changed=False`, `editor_failed=False`.
- `state["editor_iterations"]` не сбрасывается в 0 при повторном вызове.

**Mock `_call_llm`.** `AsyncMock` с `return_value` или `side_effect` для разных сценариев.

**Фикстура `autouse` для `unload_hf_llm`.** В `conftest.py`:

```python
@pytest.fixture(autouse=True)
def clean_hf_models():
    yield
    from backend.utils.hf_llm import unload_hf_llm
    unload_hf_llm()
```

Без неё `Critic(engine="huggingface")` в тесте оставит singleton в `_instances`, следующий тест получит ту же модель, `pytest` съест RAM.

---

## 5.11. Финальная проверка

```powershell
python -m poetry run pytest tests/unit/test_base_agent.py tests/unit/test_supervisor.py tests/unit/test_validator.py tests/unit/test_critic.py tests/unit/test_editor.py -v
```

Все зелёные. Плюс прогон всех предыдущих:

```powershell
python -m poetry run pytest tests/ -v
```

Финальный чек-лист:

- [ ] `BaseAgent.__abstractmethods__` содержит `process`.
- [ ] `_resolve_engine` возвращает `"ollama"` или `"huggingface"`, никогда `"auto"`.
- [ ] `_call_llm` под `@retry(max_attempts=3, delay=1.0)` и пиннит `seed=42`.
- [ ] `_adaptive_budget`: 3/2/1/0 по правилам.
- [ ] `_is_improvement`: регрессия → `False`, улучшение → `True`, равные → `False`.
- [ ] Rollback восстанавливает state, `reasoning_log` не откатывается.
- [ ] Critic fallback: `overall_score=None`, `failed=True`.
- [ ] Editor skip при `critic_failed` и при отсутствии issues.
- [ ] `pytest` — зелёный.

Коммит и merge:

```powershell
git add -A
git commit -m "feat(week-5): complete multi-agent system (Supervisor + 3 agents)"
git push -u origin week-5
```

---

## Что дальше

В Этапе 6 — сбор и импорт эталонных данных (≥ 1000 пар TM, ≥ 500 глоссария) через `scripts/eval/prepare_eval_datasets.py` (HF) и `scripts/import/add_glossary.py` + `add_tm.py`.

**В Этапе 8** — **удалить заглушки `_call_llm`** в `Critic` и `Editor`. `_call_llm` берётся из `Agent` — реальный вызов через `get_ollama_client` (Ollama) или `hf_llm.generate` (HF). Это единственное расхождение Этапа 5 с финальным кодом.

`Supervisor` и `Validator` от заглушек не зависят — `Validator` не вызывает LLM, `Supervisor` использует `Critic`/`Editor` через их публичный `process`. После удаления заглушек `Supervisor` автоматически начнёт работать с реальной LLM.